In [ ]:
!pip install -q "transformers==4.53.2" "huggingface_hub==0.33.2" "trl==0.19.1" "peft==0.16.0" "accelerate==1.8.1" "bitsandbytes==0.46.1" "datasets==3.6.0" "scikit-learn==1.7.2"

In [ ]:
import torch, random, numpy as np, json, pickle, gc, math, os, glob
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

_f = glob.glob('/kaggle/input/**/kid_adult.jsonl', recursive=True)
assert _f, "Датасет не подключён: не найден kid_adult.jsonl под /kaggle/input"
base_data = os.path.dirname(_f[0])
base_metrics = os.path.join(os.path.dirname(base_data), 'metrics')
print("данные:", base_data)

MODEL = "Qwen/Qwen3-4B-Instruct-2507"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
bnb_bf16 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
print("модель:", model.config.model_type, "| слоёв:", model.config.num_hidden_layers)

In [ ]:
tests = [json.loads(l) for l in open(f'{base_data}/public_test_style.jsonl')]

def generate(prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

print("тестов стиля:", len(tests))

In [ ]:
from scipy.sparse import hstack
with open(f'{base_metrics}/style_clf.pkl', 'rb') as f:
    style_clf = pickle.load(f)
clf = style_clf['clf']
v1, v2 = style_clf['vecs']

def proba_all(texts):
    X = hstack([v1.transform(texts), v2.transform(texts)])
    return clf.predict_proba(X)

print("classes_:", clf.classes_)

In [ ]:
from datasets import Dataset
from peft import LoraConfig

rows = [json.loads(l) for l in open(f'{base_data}/kid_adult.jsonl')]
train = Dataset.from_list([
    {"messages": [
        {"role": "user", "content": r["prompt"]},
        {"role": "assistant", "content": r["kid"]},
    ]} for r in rows])

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
print(train)

In [ ]:
from trl import SFTConfig, SFTTrainer
model.config.use_cache = False
cfg = SFTConfig(output_dir="sft_out", per_device_train_batch_size=2,
    gradient_accumulation_steps=4, num_train_epochs=1, learning_rate=2e-4,
    logging_steps=10, save_strategy="no", fp16=True, max_length=1024,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=42, report_to="none")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train,
    peft_config=lora, processing_class=tokenizer)
trainer.train()

In [ ]:
model = trainer.model
model.eval(); model.config.use_cache = True
answers = [generate(t["prompt"]) for t in tests]
probs = proba_all(answers)[:, 1]
mean_p = float(np.mean(probs))
print("=== ЗАДАЧА 1 — P_simple:", round(mean_p, 4), "===")

In [ ]:
dpo_train = Dataset.from_list([
    {"prompt":   [{"role": "user", "content": r["prompt"]}],
     "chosen":   [{"role": "assistant", "content": r["kid"]}],
     "rejected": [{"role": "assistant", "content": r["adult"]}]}
    for r in rows])
print(dpo_train)

In [ ]:
from trl import DPOConfig, DPOTrainer
model.config.use_cache = False
dpo_cfg = DPOConfig(output_dir="dpo_out", per_device_train_batch_size=1,
    gradient_accumulation_steps=8, num_train_epochs=1, learning_rate=5e-6, beta=0.1,
    logging_steps=10, save_strategy="no", fp16=True, max_length=1024, max_prompt_length=400,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=42, report_to="none")
dpo_trainer = DPOTrainer(model=model, args=dpo_cfg, train_dataset=dpo_train,
    processing_class=tokenizer)
dpo_trainer.train()

In [ ]:
model = dpo_trainer.model
model.eval(); model.config.use_cache = True
answers_dpo = [generate(t["prompt"]) for t in tests]
probs_dpo = proba_all(answers_dpo)[:, 1]
mean_p_dpo = float(np.mean(probs_dpo))
print("=== ЗАДАЧА 2 — P_simple (после DPO):", round(mean_p_dpo, 4), "===")

In [ ]:
del trainer, dpo_trainer, model
gc.collect(); torch.cuda.empty_cache()
print("память:", round(torch.cuda.memory_allocated()/1e9, 2), "ГБ")

In [ ]:
from transformers import AutoModelForSequenceClassification
rm_rows = [json.loads(l) for l in open(f'{base_data}/good_bad.jsonl')]
rm_train = Dataset.from_list([
    {"chosen":   [{"role": "user", "content": r["instruction"]},
                  {"role": "assistant", "content": r["chosen"]}],
     "rejected": [{"role": "user", "content": r["instruction"]},
                  {"role": "assistant", "content": r["rejected"]}]}
    for r in rm_rows]).shuffle(seed=42).select(range(700))

rm_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=1, quantization_config=bnb_bf16, device_map="auto", torch_dtype=torch.bfloat16)
rm_model.config.pad_token_id = tokenizer.pad_token_id
rm_model.config.use_cache = False
print("reward-модель, пар:", len(rm_train))

In [ ]:
from trl import RewardConfig, RewardTrainer
rm_lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
rm_cfg = RewardConfig(output_dir="rm_out", per_device_train_batch_size=1,
    gradient_accumulation_steps=8, num_train_epochs=1, learning_rate=1e-5,
    warmup_ratio=0.1, max_grad_norm=0.5, center_rewards_coefficient=0.01,
    logging_steps=20, save_strategy="no", bf16=True, fp16=False, max_length=448,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=42, report_to="none")
rm_trainer = RewardTrainer(model=rm_model, args=rm_cfg, train_dataset=rm_train,
    processing_class=tokenizer, peft_config=rm_lora)
rm_trainer.train()

In [ ]:
rm = rm_trainer.model; rm.eval()
def rm_score(prompt, answer):
    m = [{"role": "user", "content": prompt},
         {"role": "assistant", "content": answer}]
    inp = tokenizer.apply_chat_template(m, tokenize=True, return_tensors="pt", return_dict=True).to(rm.device)
    with torch.no_grad(): out = rm(**inp)
    return float(out.logits.flatten()[0])

tests_q = [json.loads(l) for l in open(f'{base_data}/public_test_quality.jsonl')]
correct = sum(1 for t in tests_q
              if rm_score(t["prompt"], t["chosen"]) > rm_score(t["prompt"], t["rejected"]))
rm_acc = correct / len(tests_q)
print("=== ЗАДАЧА 3 — accuracy:", round(rm_acc, 4), "===")

In [ ]:
del rm_trainer, rm_model, rm
gc.collect(); torch.cuda.empty_cache()
print("память:", round(torch.cuda.memory_allocated()/1e9, 2), "ГБ")


In [ ]:
def mean_logprob(model, prompt, answer):
    user_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True, return_tensors="pt").to(model.device)
    full_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}],
        return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(full_ids).logits
    logp = torch.log_softmax(logits[0, :-1], dim=-1)
    ans_ids = full_ids[0, user_ids.shape[1]:]
    tok_logp = logp[user_ids.shape[1]-1: full_ids.shape[1]-1].gather(1, ans_ids.unsqueeze(1)).squeeze(1)
    return tok_logp.mean().item()

def pref_accuracy(model):
    correct = sum(1 for t in tests_q
                  if mean_logprob(model, t["prompt"], t["chosen"]) > mean_logprob(model, t["prompt"], t["rejected"]))
    return correct / len(tests_q)

print("метрика готова: implicit-preference accuracy (length-normalized logprob)")


In [ ]:
dpo_train_q = Dataset.from_list([
    {"prompt":   [{"role": "user", "content": r["instruction"]}],
     "chosen":   [{"role": "assistant", "content": r["chosen"]}],
     "rejected": [{"role": "assistant", "content": r["rejected"]}]}
    for r in rm_rows])

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
print(dpo_train_q)


In [ ]:
from trl import DPOConfig, DPOTrainer
model.config.use_cache = False
dpo_q_cfg = DPOConfig(output_dir="dpo_q_out", per_device_train_batch_size=1,
    gradient_accumulation_steps=8, num_train_epochs=1, learning_rate=5e-6, beta=0.1,
    logging_steps=20, save_strategy="no", fp16=True, max_length=448, max_prompt_length=128,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=42, report_to="none")
dpo_q_trainer = DPOTrainer(model=model, args=dpo_q_cfg, train_dataset=dpo_train_q,
    peft_config=lora, processing_class=tokenizer)
dpo_q_trainer.train()


In [ ]:
model = dpo_q_trainer.model
model.eval(); model.config.use_cache = True
dpo_q_acc = pref_accuracy(model)
print("=== ЗАДАЧА 4 — accuracy:", round(dpo_q_acc, 4), "===")


In [ ]:
print("="*45)
print("Задача 1 (SFT)   P_simple =", round(mean_p, 4))
print("Задача 2 (DPO)   P_simple =", round(mean_p_dpo, 4))
print("Задача 3 (RM)    accuracy =", round(rm_acc, 4))
print("Задача 4 (DPO)   accuracy =", round(dpo_q_acc, 4))
